Module 7: Learning from Pre-Built Models - Assignment

Problem Statement:
The categorization of images into distinct classes is a pervasive task in computer
vision, and it has a wide array of applications, including in pet identification and
animal monitoring systems. In this assignment, we aim to develop a model
capable of accurately distinguishing between cat and dog images. Instead of
building a convolutional neural network (CNN) from scratch, we will leverage
transfer learning using the VGG16 model, a pre-trained model on the ImageNet
dataset. VGG16 is renowned for its effectiveness in image recognition tasks, but
it does not have broad categories for cats and dogs. Therefore, we will utilize the
convolutional (Conv) layers of VGG16 for feature extraction and add custom fully
connected (Cat) layers for the classification task

Objectives:
Utilize VGG16 for Feature Extraction:
● Employ the VGG16 model, excluding its top layers, to serve as a feature
extractor for cat and dog images.
● Ensure the input images are of the correct size (150x150) and preprocessed
appropriately to match VGG16’s requirements.

Data Preprocessing and Augmentation:
Implement image data generators for real-time data augmentation, ensuring a robost and varied dataset for training 
the classification layers.

Build and Train the classification Model:
● Add custom fully connected layers on top of the VGG16 model for classification task
● Freeze the convolutional layers VGG16 to retain the pre-trained featureand only train the added classification layers.

Model Compilation and Training:
● Compile the model using stochastic gradient descent, categorical
cross-entropy as the loss function, and accuracy as the evaluation metric.
● Train the model using the training data, and validate its performance using
a validation set.

Evaluate and Test the Model:
● Assess the model’s performance based on its accuracy in classifying
images into cat or dog categories.
● Implement a prediction function to classify new images, providing the
predicted category and the associated confidence level

Prepare Dataset

In [246]:
import os

# Define the base directory for the dataset
base_dir = 'cats_and_dogs_filtered'

# Create the base directory if it doesn't exist
if not os.path.exists(base_dir):
    os.makedirs(base_dir)
    print(f"Created base directory: {base_dir}")
else:
    print(f"Base directory already exists: {base_dir}")

# Define subdirectories for train, validation, and test
sub_dirs = ['train', 'validation', 'test']

# Define categories for each subdirectory
categories = ['cats', 'dogs']

# Create the full directory structure
for sub_dir in sub_dirs:
    for category in categories:
        path = os.path.join(base_dir, sub_dir, category)
        if not os.path.exists(path):
            os.makedirs(path)
            print(f"Created directory: {path}")
        else:
            print(f"Directory already exists: {path}")

print("Dataset directory structure set up successfully!")

Base directory already exists: cats_and_dogs_filtered
Directory already exists: cats_and_dogs_filtered\train\cats
Directory already exists: cats_and_dogs_filtered\train\dogs
Directory already exists: cats_and_dogs_filtered\validation\cats
Directory already exists: cats_and_dogs_filtered\validation\dogs
Directory already exists: cats_and_dogs_filtered\test\cats
Directory already exists: cats_and_dogs_filtered\test\dogs
Dataset directory structure set up successfully!


Data Preprocessing and Augmentation

In [248]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
from PIL import Image

data_dir = './cats_and_dogs_filtered'
print("Starting data cleanup...")
removed_count = 0
for root, dirs, files in os.walk(data_dir):
    for filename in files:
        # Check for image files
        if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            file_path = os.path.join(root, filename)
            try:
                img = Image.open(file_path)
                img.verify()  # Verifies the file is actually an image
            except (IOError, SyntaxError, Image.UnidentifiedImageError):
                print(f"Removing corrupt file: {file_path}")
                os.remove(file_path)
                removed_count += 1

print(f"Cleanup complete. Removed {removed_count} bad files.")

# Define image dimensions and batch size
IMAGE_SIZE = (150, 150)
BATCH_SIZE =32

# --- Training Data Generator with Augmentation ---
train_datagen = ImageDataGenerator(
    rescale=1./255, # Normalize pixel values to [0, 1]
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# --- Validation and Test Data Generator (only rescaling) ---
# Validation and test data should not be augmented, only rescaled/preprocessed
validation_test_datagen = ImageDataGenerator(rescale=1./255)

# --- Create Generators from Directories ---
# Base directory was already defined as 'cats_and_dogs_filtered'

train_generator = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

validation_generator = validation_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'validation'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = validation_test_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False # Keep data in order for evaluation
)

print("Data generators created successfully!")
print(f"Number of training samples: {train_generator.samples}")
print(f"Number of validation samples: {validation_generator.samples}")
print(f"Number of test samples: {test_generator.samples}")

Starting data cleanup...
Cleanup complete. Removed 0 bad files.
Found 10001 images belonging to 2 classes.
Found 90 images belonging to 2 classes.
Found 41 images belonging to 2 classes.
Data generators created successfully!
Number of training samples: 10001
Number of validation samples: 90
Number of test samples: 41


Load VGG16 Model for Feature Extraction

In [250]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model

# Load the VGG16 model without the top (fully connected) layers
vgg16_base = VGG16(
    weights='imagenet',       # Use pre-trained weights from ImageNet
    include_top=False,        # Exclude the 3 fully-connected layers at the top of the network
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3) # Specify the input shape
)

# Freeze the convolutional layers of VGG16
for layer in vgg16_base.layers:
    layer.trainable = False

print("VGG16 base model loaded and convolutional layers frozen successfully!")
print(f"Number of layers in VGG16 base: {len(vgg16_base.layers)}")
print(f"Are VGG16 base layers trainable? {vgg16_base.trainable}")

VGG16 base model loaded and convolutional layers frozen successfully!
Number of layers in VGG16 base: 19
Are VGG16 base layers trainable? True


Build Custom Classification Head

In [252]:
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras import Sequential

# Get the output of the vgg16_base model
x = vgg16_base.output

# Add custom fully connected layers
x = Flatten()(x)
x = Dense(256, activation='relu')(x) # First Dense layer with ReLU activation
x = Dropout(0.5)(x) # Dropout to prevent overfitting
x = Dense(128, activation='relu')(x) # Second Dense layer with ReLU activation
x = Dropout(0.5)(x) # Dropout to prevent overfitting
predictions = Dense(1, activation='sigmoid')(x) # Final Dense layer for binary classification

# Create the complete model by connecting the vgg16_base input to the custom classification head
model = Model(inputs=vgg16_base.input, outputs=predictions)

print("Custom classification head added and model compiled successfully!")
model.summary()

Custom classification head added and model compiled successfully!


Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_13 (InputLayer)     │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 150, 150, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 150, 150, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 75, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 75, 75, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 75, 75, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 37, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 37, 37, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 18, 18, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 18, 18, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_13 (Flatten)            │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_26 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_27 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 1)              │           12

 Total params: 16,845,121 (64.26 MB)

 Trainable params: 2,130,433 (8.13 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

Compile the Model

In [254]:
from tensorflow.keras.optimizers import Adam

# Compile the model
model.compile(
    optimizer=Adam(learning_rate=1e-5), # Use Adam optimizer with a small learning rate
    loss='binary_crossentropy',          # Binary cross-entropy for binary classification
    metrics=['accuracy']                 # Track accuracy during training
)

print("Model compiled successfully!")

Model compiled successfully!


Train the Model

In [256]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Define the number of epochs (can be higher, EarlyStopping will manage it)
EPOCHS = 5

# 1. Instantiate ModelCheckpoint
checkpoint_filepath = 'best_model.weights.h5'
checkpoint_callback = ModelCheckpoint(
    filepath=checkpoint_filepath,
    monitor='val_accuracy', # Monitor validation accuracy
    save_best_only=True,   # Save only the best model weights
    mode='max',            # Maximize validation accuracy
    verbose=1              # Log when a better model is saved
)

# 2. Instantiate EarlyStopping
early_stopping_callback = EarlyStopping(
    monitor='val_loss',     # Monitor validation loss
    patience=10,            # Stop if validation loss doesn't improve for 10 epochs
    restore_best_weights=True, # Restore model weights from the epoch with the best value of the monitored quantity.
    verbose=1
)

# Calculate steps per epoch for training and validation
steps_per_epoch = train_generator.samples // BATCH_SIZE
validation_steps = validation_generator.samples // BATCH_SIZE

# Ensure there's at least one step if samples are less than BATCH_SIZE but not zero
if train_generator.samples % BATCH_SIZE > 0 and train_generator.samples > 0:
    steps_per_epoch += 1
if validation_generator.samples % BATCH_SIZE > 0 and validation_generator.samples > 0:
    validation_steps += 1

# Check if generators have samples before training
if train_generator.samples == 0:
    print("Error: Training data generator is empty. Please ensure image files are placed in the 'cats_and_dogs_filtered/train' directories.")
elif validation_generator.samples == 0:
    print("Error: Validation data generator is empty. Please ensure image files are placed in the 'cats_and_dogs_filtered/validation' directories.")
else:
    # Train the model
    history = model.fit(
        train_generator,
        steps_per_epoch=steps_per_epoch,
        epochs=EPOCHS,
        validation_data=validation_generator,
        validation_steps=validation_steps,
        callbacks=[checkpoint_callback, early_stopping_callback]
    )

    print("Model training initiated with ModelCheckpoint and EarlyStopping.")

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.5197 - loss: 0.7938
Epoch 1: val_accuracy improved from -inf to 0.88889, saving model to best_model.weights.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 1938s 6s/step - accuracy: 0.5198 - loss: 0.7936 - val_accuracy: 0.8889 - val_loss: 0.4661
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.6315 - loss: 0.6340
Epoch 2: val_accuracy improved from 0.88889 to 0.93333, saving model to best_model.weights.h5


313/313 ━━━━━━━━━━━━━━━━━━━━ 1968s 6s/step - accuracy: 0.6315 - loss: 0.6339 - val_accuracy: 0.9333 - val_loss: 0.3408
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.7086 - loss: 0.5684
Epoch 3: val_accuracy did not improve from 0.93333
313/313 ━━━━━━━━━━━━━━━━━━━━ 1971s 6s/step - accuracy: 0.7086 - loss: 0.5683 - val_accuracy: 0.9333 - val_loss: 0.2648
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.7435 - loss: 0.5186
Epoch 4: val_accuracy did not improve from 0.93333
313/313 ━━━━━━━━━━━━━━━━━━━━ 2002s 6s/step - accuracy: 0.7435 - loss: 0.5186 - val_accuracy: 0.9222 - val_loss: 0.2375
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 6s/step - accuracy: 0.7569 - loss: 0.4913
Epoch 5: val_accuracy did not improve from 0.93333
313/313 ━━━━━━━━━━━━━━━━━━━━ 2004s 6s/step - accuracy: 0.7570 - loss: 0.4913 - val_accuracy: 0.9333 - val_loss: 0.2088
Restoring model weights from the end of the best epoch: 5.
Model training initiated with ModelCheckpoint and EarlyStopp

Acquire and Organize Dataset

In [286]:
import os
import zipfile
import shutil
import random

# Define paths and ratios
zip_file_path = 'train.zip'  # Assuming train.zip is uploaded to /content/
data_dir = 'cats_and_dogs_filtered'
original_dataset_dir = 'train' # Directory created after unzipping train.zip
print(f"print zip_file_path: {zip_file_path}")
print(f"print data_dir: {data_dir}")
print(f"print original_dataset_dir: {original_dataset_dir}")
train_ratio = 0.8
validation_ratio = 0.1
test_ratio = 0.1

# 1. Unzip the downloaded dataset
if os.path.exists(zip_file_path):
    print(f"Unzipping {zip_file_path}...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall('.')
    print("Unzipping complete.")
else:
    print(f"Error: {zip_file_path} not found. Please ensure the 'train.zip' file is uploaded to the current directory.")

# Ensure the original_dataset_dir exists after unzipping
if not os.path.exists(original_dataset_dir):
    print(f"Error: '{original_dataset_dir}' directory not found after unzipping. Please check the zip file content.")
else:
    print(f"Found original dataset directory: {original_dataset_dir}")

    # Get all image filenames
    all_fnames = [fname for fname in os.listdir(original_dataset_dir) if fname.endswith('.jpg')]
    random.shuffle(all_fnames)

    # Calculate split points
    total_images = len(all_fnames)
    train_split = int(total_images * train_ratio)
    validation_split = int(total_images * validation_ratio)

    train_fnames = all_fnames[:train_split]
    validation_fnames = all_fnames[train_split : train_split + validation_split]
    test_fnames = all_fnames[train_split + validation_split :]

    print(f"Total images: {total_images}")
    print(f"Train images: {len(train_fnames)}")
    print(f"Validation images: {len(validation_fnames)}")
    print(f"Test images: {len(test_fnames)}")

    # Helper function to copy images
    def copy_images(fnames, dest_dir):
        for fname in fnames:
            src = os.path.join(original_dataset_dir, fname)
            if 'cat' in fname:
                dst = os.path.join(data_dir, dest_dir, 'cats', fname)
            elif 'dog' in fname:
                dst = os.path.join(data_dir, dest_dir, 'dogs', fname)
            else:
                continue # Skip files not clearly cat or dog
            shutil.copyfile(src, dst)

    # Copy images to their respective directories
    print("Copying training images...")
    copy_images(train_fnames, 'train')
    print("Copying validation images...")
    copy_images(validation_fnames, 'validation')
    print("Copying test images...")
    copy_images(test_fnames, 'test')

    print("Dataset organization complete!")

print zip_file_path: train.zip
print data_dir: cats_and_dogs_filtered
print original_dataset_dir: train
Error: train.zip not found. Please ensure the 'train.zip' file is uploaded to the current directory.
Found original dataset directory: train
Total images: 25000
Train images: 20000
Validation images: 2500
Test images: 2500
Copying training images...
Copying validation images...
Copying test images...
Dataset organization complete!
